# Neural Encoder Interpretation

## Imports and Setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

import h5py
import numpy as np
import pandas as pd
import torch

from src.classes import *
from src.helper_functions import *
from src.plot_functions import *

SEED = 1
seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Configuration

In [ ]:
Conf = DotDict({
    "run_id": 1,
    "seed": SEED,
    "paths": {
        "data": r"D:\Research\Project\THINGS\data\data.mat",
        "metadata": r"D:\Research\Project\THINGS\data\data_metadata.csv",
    },
    "device": device,
    "data": {
        "bin_size": 0.01,
    },
    "training": {
        "batch_size": 16,
        "max_epoch": 100,
        "min_delta": 1e-5,
        "patience": 5,
    },
    "model_type": {
        "name": "cnn_transformer",
        "cnn_transformer": {
            "cnn_n_hidden": 4,
            "cnn_n_layers": 1,
            "positional_n_hidden": 16,
            "transformer_n_hidden": 32,
            "transformer_n_heads": 2,
            "transformer_n_layers": 1,
            "dropout": 0.2,
            "transformer_nonlinearity": "gelu",
        },
    },
    "optimization": {
        "Adam": {
            "lr": 0.0001,
            "weight_decay": 0,
        },
        "Reduce": {
            "factor": 0.5,
            "patience": 2,
            "min_lr": 1e-10,
        },
        "entropy_lambda": 0.0,
        "coverage_lambda": 0.0,
    },
})

## Data Preparation

In [ ]:
with h5py.File(Conf.paths.data, "r") as f:
    g = f["data"]

    allmat = np.array(g["ALLMAT"]).T
    spikes = np.array(g["ALLMUA"]).transpose(1, 2, 0)
    images = np.array(g["IMAGES"]).transpose(0, 2, 1, 3)
    bin_times = np.array(g["tb"]).ravel()
    electrode_names = np.array(g["selectedElectrodes"]).ravel().astype(int)
    mapping = np.array(g["mapping"]).ravel().astype(int)

train_idx = allmat[:, 1].astype(int)
test_idx = allmat[:, 2].astype(int)
rep = allmat[:, 3].astype(int)
count = allmat[:, 4].astype(int)
day = allmat[:, 5].astype(int)

metadata = pd.read_csv(Conf.paths.metadata)
image_names = metadata["image_name"].str.rsplit(".", n=1).str[0].to_numpy()

electrode_roi = np.empty(len(electrode_names), dtype="<U2")
electrode_roi[electrode_names <= 512] = "V1"
electrode_roi[(electrode_names >= 513) & (electrode_names <= 832)] = "IT"
electrode_roi[electrode_names >= 833] = "V4"

In [ ]:
trials_to_remove, electrodes_to_remove = plot_spike_summary(
    spikes=spikes,
    file_name="spike-summary-before",
    file_path="./plot/spikes",
    metric="outlier_rate",
    k=5.0,
    electrode_threshold=0.02,
    trial_threshold=0.02,
    electrode_names=electrode_names,
)

for electrode_idx in electrodes_to_remove:
    plot_electrode_spikes(
        electrode_spikes=spikes[:, electrode_idx, :],
        title=f"Electrode {electrode_names[electrode_idx]}",
        file_name=f"electrode-{electrode_idx}",
        file_path="./plot/spikes/electrodes_to_remove/",
        bin_times=bin_times,
    )

In [ ]:
electrodes_to_remove = [172, 355, 462, 723, 731, 755, 767, 852, 863]

clean_trials_mask = np.ones(spikes.shape[0], dtype=bool)
clean_trials_mask[trials_to_remove] = False

spike_trials_mask_finite = np.isfinite(spikes).any(axis=(1, 2))
image_trials_mask_finite = np.isfinite(images).any(axis=(1, 2, 3))
finite_trials_mask = spike_trials_mask_finite & image_trials_mask_finite

keep_trials_mask = clean_trials_mask & finite_trials_mask


clean_electrodes_mask = np.ones(spikes.shape[1], dtype=bool)
clean_electrodes_mask[electrodes_to_remove] = False

nonnegative_electrodes_mask = (spikes >= 0).all(axis=(0, 2))

keep_electrodes_mask = clean_electrodes_mask & nonnegative_electrodes_mask


spikes = spikes[keep_trials_mask][:, keep_electrodes_mask, :]
images = images[keep_trials_mask]
allmat = allmat[keep_trials_mask]

train_idx = train_idx[keep_trials_mask]
test_idx = test_idx[keep_trials_mask]
rep = rep[keep_trials_mask]
count = count[keep_trials_mask]
day = day[keep_trials_mask]
image_names = image_names[keep_trials_mask]

electrode_names = electrode_names[keep_electrodes_mask]
electrode_roi = electrode_roi[keep_electrodes_mask]

n_trials, n_electrodes, n_bins = spikes.shape
n_trials, n_pixels, n_pixels, n_channels = images.shape

In [ ]:
Conf.data.n_trials = n_trials
Conf.data.n_electrodes = n_electrodes
Conf.data.n_bins = len(bin_times)
Conf.data.n_pixels = n_pixels
Conf.data.n_channels = n_channels

electrode_names_V1 = electrode_names[electrode_roi == 'V1']
spikes_V1 = spikes[:, electrode_roi == 'V1', :]
n_electrodes_V1 = len(electrode_names_V1)
Conf_V1 = copy.deepcopy(Conf)
Conf.data.n_electrodes = n_electrodes_V1

electrode_names_V4 = electrode_names[electrode_roi == 'V4']
spikes_V4 = spikes[:, electrode_roi == 'V4', :]
n_electrodes_V4 = len(electrode_names_V4)
Conf_V4 = copy.deepcopy(Conf)
Conf.data.n_electrodes = n_electrodes_V4

electrode_names_IT = electrode_names[electrode_roi == 'IT']
spikes_IT = spikes[:, electrode_roi == 'IT', :]
n_electrodes_IT = len(electrode_names_IT)
Conf_IT = copy.deepcopy(Conf)
Conf.data.n_electrodes = n_electrodes_IT

In [ ]:
train_dataset_V1, test_dataset_V1, image_mean, image_std, Y_mean_V1, Y_std_V1 = prepare_dataset(Conf, images, spikes_V1, train_idx, test_idx)
train_loader_V1, test_loader_V1, loader_generators_V1 = prepare_loader(Conf, train_dataset_V1, test_dataset_V1)

train_dataset_V4, test_dataset_V4, image_mean, image_std, Y_mean, Y_std = prepare_dataset(Conf, images, spikes_V4, train_idx, test_idx)
train_loader_V4, test_loader_V4, loader_generators_V4 = prepare_loader(Conf, train_dataset_V4, test_dataset_V4)

train_dataset_IT, test_dataset_IT, image_mean, image_std, Y_meanIT, Y_stdIT = prepare_dataset(Conf, images, spikes_IT, train_idx, test_idx)
train_loader_IT, test_loader_IT, loader_generators_IT = prepare_loader(Conf, train_dataset_IT, test_dataset_IT)

In [ ]:
electrode_idx = 0

plot_electrode_spikes(
    electrode_spikes=spikes[train_idx > 0, electrode_idx, :],
    title=f"Electrode {electrode_names[electrode_idx]} in V1",
    file_name=f"electrode-{electrode_idx}-V1-raw",
    file_path=f"./plot/spikes/electrodes/",
    bin_times=bin_times,
)

plot_electrode_spikes(
    electrode_spikes=train_dataset_V1[:][1][:, :, electrode_idx].numpy(),
    title=f"Electrode {electrode_names[electrode_idx]} in V1",
    file_name=f"electrode-{electrode_idx}-V1-prep",
    file_path=f"./plot/spikes/electrodes/",
    bin_times=bin_times,
)

In [ ]:
trial_idx = 56

plot_image(
    image=images[train_idx > 0][trial_idx],
    title=f"Image {image_names[trial_idx]}",
    file_name=f"image-{trial_idx}-raw",
    file_path="./plot/images/",
    vmin=[0, 0, 0],
    vmax=[255, 255, 255],
)

plot_image(
    image=train_dataset_V1[trial_idx][0].permute(1, 2, 0).numpy(),
    title=f"Image {image_names[trial_idx]}",
    file_name=f"image-{trial_idx}-prep",
    file_path="./plot/images/",
    vmin=(0 - image_mean.squeeze().numpy()) / image_std.squeeze().numpy(),
    vmax=(255 - image_mean.squeeze().numpy()) / image_std.squeeze().numpy(),
)

## Model Training

In [ ]:
cnn_n_layers_list = [1, 2, 3]

trainer_V1_list = []
lit_model_V1_list = []

for cnn_n_layers in cnn_n_layers_list:
    Conf_V1 = copy.deepcopy(Conf)
    Conf_V1.model_type.cnn_transformer.cnn_n_layers = cnn_n_layers
    Conf_V1.data.n_electrodes = n_electrodes_V1

    trainer, lit_model = build_lit_model(Conf_V1, loader_generators_V1, "cnn_transformer", True)
    trainer.fit(lit_model, train_loader_V1, test_loader_V1)

    trainer_V1_list.append(trainer)
    lit_model_V1_list.append(lit_model)

In [ ]:
cnn_n_layers_list = [1, 2, 3, 4]

trainer_IT_list = []
lit_model_IT_list = []

for cnn_n_layers in cnn_n_layers_list:
    Conf_IT = copy.deepcopy(Conf)
    Conf_IT.model_type.cnn_transformer.cnn_n_layers = cnn_n_layers
    Conf_IT.data.n_electrodes = n_electrodes_IT

    trainer, lit_model = build_lit_model(Conf_IT, loader_generators_IT, "cnn_transformer", True)
    trainer.fit(lit_model, train_loader_IT, test_loader_IT)

    trainer_IT_list.append(trainer)
    lit_model_IT_list.append(lit_model)

In [ ]:
trainer_V4_list = []
lit_model_V4_list = []

for cnn_n_layers in cnn_n_layers_list:
    Conf_V4 = copy.deepcopy(Conf)
    Conf_V4.model_type.cnn_transformer.cnn_n_layers = cnn_n_layers
    Conf_V4.data.n_electrodes = n_electrodes_V4

    trainer, lit_model = build_lit_model(Conf_V4, loader_generators_V4, "cnn_transformer", True)
    trainer.fit(lit_model, train_loader_V4, test_loader_V4)

    trainer_V4_list.append(trainer)
    lit_model_V4_list.append(lit_model)

In [ ]:
plot_training_history(
    trainer=trainer,
    title=None,
    file_name="training-history",
    file_path="./plot/model/",
)

## Model Evaluation

In [ ]:
Y_train, Y_hat_train = predict_loader(Conf, lit_model, train_loader, Y_mean, Y_std)
Y_test, Y_hat_test = predict_loader(Conf, lit_model, test_loader, Y_mean, Y_std)

In [ ]:
correlation_test, r2_test, mse_test = compute_metrics(Conf, Y_test, Y_hat_test)

In [ ]:
for roi_name in ["V1", "IT", "V4"]:
    plot_electrode_metrics(
        correlation_electrode=correlation_test[electrode_roi == roi_name],
        r2_electrode=r2_test[electrode_roi == roi_name],
        mse_electrode=mse_test[electrode_roi == roi_name],
        bin_times=bin_times,
        title=f"{roi_name}",
        file_name=f"roi_{roi_name}_metrics",
        file_path="./plot/model/roi_metrics",
        show=False,
    )

In [ ]:
test_idx_test = test_idx[test_idx > 0]

test_image_idx = np.unique(test_idx_test)

bin_indices = np.where((bin_times >= 0) & (bin_times <= 200))[0]

correlation_test_image = []
r2_test_image = []
mse_test_image = []

test_image_names = []

for image_idx in test_image_idx:
    image_mask = test_idx_test == image_idx

    Y_image = Y_test[image_mask].mean(axis=0)
    Y_hat_image = Y_hat_test[image_mask].mean(axis=0)

    correlation_electrode = np.zeros(Conf.data.n_electrodes)
    r2_electrode = np.zeros(Conf.data.n_electrodes)
    mse_electrode = np.zeros(Conf.data.n_electrodes)

    for electrode_idx in range(Conf.data.n_electrodes):
        y = Y_image[bin_indices, electrode_idx]
        y_hat = Y_hat_image[bin_indices, electrode_idx]

        correlation_electrode[electrode_idx] = compute_correlation(y, y_hat)
        r2_electrode[electrode_idx] = compute_r2(y, y_hat)
        mse_electrode[electrode_idx] = compute_mse(y, y_hat)

    correlation_test_image.append(correlation_electrode)
    r2_test_image.append(r2_electrode)
    mse_test_image.append(mse_electrode)

    test_image_names.append(image_names[test_idx == image_idx][0])

correlation_test_image = np.array(correlation_test_image)
r2_test_image = np.array(r2_test_image)
mse_test_image = np.array(mse_test_image)

In [ ]:
for roi_name in ["V1", "IT", "V4"]:
    plot_image_metrics(
        correlation_image=correlation_test_image[:, electrode_roi == roi_name],
        r2_image=r2_test_image[:, electrode_roi == roi_name],
        mse_image=mse_test_image[:, electrode_roi == roi_name],
        image_names = image_names,
        title=f"{roi_name}",
        file_name=f"roi_{roi_name}_metrics",
        file_path="./plot/model/image_metrics",
        show=False,
    )

In [ ]:
image_label = "monkey/monkey_18n.jpg"

image_idx = test_idx[image_names == image_label][0]

image_mask = test_idx_test == image_idx

Y_image = Y_test[image_mask].mean(axis=0)
Y_hat_image = Y_hat_test[image_mask].mean(axis=0)

correlation_electrode = np.zeros(Conf.data.n_electrodes)
r2_electrode = np.zeros(Conf.data.n_electrodes)
mse_electrode = np.zeros(Conf.data.n_electrodes)

for electrode_idx in range(Conf.data.n_electrodes):
    y = Y_image[bin_indices, electrode_idx]
    y_hat = Y_hat_image[bin_indices, electrode_idx]

    correlation_electrode[electrode_idx] = compute_correlation(y, y_hat)
    r2_electrode[electrode_idx] = compute_r2(y, y_hat)
    mse_electrode[electrode_idx] = compute_mse(y, y_hat)



for roi_name in ["V1", "IT", "V4"]:
    roi_mask = electrode_roi == roi_name

    plot_electrode_metrics_hist(
        correlation_electrode=correlation_electrode[roi_mask],
        r2_electrode=r2_electrode[roi_mask],
        mse_electrode=mse_electrode[roi_mask],
        title=roi_name,
        file_name=f"{roi_name}_metrics_hist",
        file_path=f"./plot/model/image_metrics/{image_label}",
        show=False,
    )

## Model Interpretation

In [ ]:
bin_idx = np.argmin(np.abs(bin_times - 100))
trial_idx = np.where(image_mask)[0][0]

In [ ]:
bin_indices_baseline

In [ ]:
bin_indices_response

In [ ]:
device_old = Conf.device
Conf.device = torch.device("cpu")

lit_model = lit_model.cpu()

torch.cuda.empty_cache()

baseline_black = torch.zeros_like(test_dataset[trial_idx][0]).unsqueeze(0)
bin_indices_baseline = np.where((bin_times >= -100) & (bin_times <= -50))[0]

for sort_idx in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]:
    for roi_name in ["V1", "IT", "V4"]:
        roi_electrode_indices = np.where(electrode_roi == roi_name)[0]

        bin_indices_response = [np.argmax(np.mean(correlation_test[roi_electrode_indices], axis=0))]

        sorted_local_idx = np.argsort(correlation_electrode[roi_electrode_indices])[::-1]
        local_idx = sorted_local_idx[sort_idx]
        electrode_idx = roi_electrode_indices[local_idx]

        # attribution_maps_baseline = []
        # for bin_idx_basline in bin_indices_baseline:
        #     attribution_map = compute_attribution_map(
        #         Conf,
        #         lit_model,
        #         test_dataset[trial_idx],
        #         electrode_idx=electrode_idx,
        #         bin_idx=bin_idx_basline,
        #         method="occlusion",
        #         baseline=baseline_black.cpu(),
        #         sliding_window_shapes=(3, 10, 10),
        #         strides=(3, 2, 2),
        #     )
        #     attribution_maps_baseline.append(attribution_map)
        # attribution_maps_baseline = np.stack(attribution_maps_baseline, axis=-1)
        # baseline_mean = attribution_maps_baseline.mean(axis=3)
        # baseline_std = attribution_maps_baseline.std(axis=3)

        baseline_mean = 0
        baseline_std = 1

        for bin_idx_response in bin_indices_response:
            attribution_map = compute_attribution_map(
                Conf,
                lit_model,
                test_dataset[trial_idx],
                electrode_idx=electrode_idx,
                bin_idx=bin_idx_response,
                method="occlusion",
                baseline=baseline_black.cpu(),
                sliding_window_shapes=(3, 10, 10),
                strides=(3, 2, 2),
            )

            attribution_map_norm = (attribution_map - baseline_mean) / (baseline_std + 1e-8)
            vmax_attribution = np.abs(attribution_map_norm).max()

            plot_attribution(
                image=test_dataset[trial_idx][0].permute(1, 2, 0).numpy(),
                attribution=attribution_map_norm.transpose(1, 2, 0),
                vmax_attribution=vmax_attribution,
                vmin_attribution=-vmax_attribution,
                title=f"Occlusion Attribution of Electrode {electrode_names[electrode_idx]} in {roi_name} at Bin {bin_times[bin_idx_response]}",
                file_name=f"electrode-{electrode_idx}-bin{bin_idx_response}",
                file_path=f"./plot/model/attribution/occlusion/{roi_name}",
                show=False,
            )

Conf.device = device_old
lit_model = lit_model.to(device_old).eval()

torch.cuda.empty_cache()